# Minimal Sage Agent Run

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import rootutils
from dotenv import load_dotenv

root = rootutils.setup_root(search_from='..', indicator="pyproject.toml", pythonpath=True, cwd=True)
print(f"Root directory: {root}")
print(f".env file path: {root / '.env'}")
load_dotenv(dotenv_path= root / ".env")

In [ ]:
from pydantic import SecretStr

from langchain_openai import ChatOpenAI

from src.sage.runtime import SageRuntime
from src.sage.types import SageRuntimeConfig
from src.tools.catalog import AVAILABLE_TOOLS
from src.agent.controller import AgentController, ControllerConfig
from src.utils.console_logging import ConsoleLogger

In [ ]:
logger = ConsoleLogger()

model = ChatOpenAI(
    model=f"gpt://{os.environ.get('YANDEX_FOLDER_ID')}/gpt-oss-20b/latest",
    base_url = "https://ai.api.cloud.yandex.net/v1",
    api_key = SecretStr(os.environ.get('YANDEX_API_KEY', ".env hasn't been loaded or YANDEX_API_KEY is missing")),
    temperature = 0,
    timeout = 60,
    max_retries = 2,
)

sage_runtime_config = SageRuntimeConfig()
sage_runtime = SageRuntime(config=sage_runtime_config, logger=logger)

with open('prompts/sage_skill/v1.md', 'r') as f:
    sage_skill = f.read()

with open('prompts/system_prompt/v2.md', 'r') as f:
    system_prompt = f.read()

tools = [tool(sage_runtime, sage_skill if key == "sage_exec" else "") for key, tool in AVAILABLE_TOOLS.items()]

controller_config = ControllerConfig(
    max_steps=4,
    progress_logs=True,
    max_tool_calls=4,
)

controller = AgentController(model=model, config=controller_config, tools=tools, system_prompt=system_prompt, logger=logger)

In [ ]:
prompt = """Construct a degree-19 polynomial p(x) in C[x] such that

    X := {p(x) = p(y)} ⊂ P¹ × P¹

has at least 3 irreducible components over C,
but not all components are linear.

Additional constraints:

• p(x) must be odd  
• p(x) must be monic  
• coefficients must be real  
• the linear coefficient must be −19

Then compute p(19)."""

In [ ]:
result = controller.solve(prompt)

In [ ]:
result.tool_traces[0]["arguments"]["code"]

In [ ]:
for trace in result.tool_traces:
    print(f"Turn: {trace['turn']}\nCode:{trace['arguments']['code']}")